# Post-Quantum Cryptography — Comparative Analysis

**Research:** IME — Instituto Militar de Engenharia  
**Author:** Bernardo Pereira Pinto de Castro  
**Advisor:** Cel R/1 Anderson Fernandes Pereira dos Santos

---

This notebook presents a comparative analysis of four NIST-standardized Post-Quantum Cryptography (PQC) algorithms benchmarked under identical conditions (100 iterations, Google Colab CPU environment).

| Algorithm | Type | Family |
|-----------|------|--------|
| CRYSTALS-Kyber512 | KEM | Lattice (Module-LWE) |
| CRYSTALS-Dilithium2 | Signature | Lattice (Module-LWE/SIS) |
| Classic McEliece-6960119 | KEM | Code-based (Goppa) |
| SPHINCS+-SHAKE-256s | Signature | Hash-based |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# ── Shared style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'axes.titlecolor':  '#f0f6fc',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linewidth':   0.8,
    'font.family':      'monospace',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

# Algorithm labels and color palette
ALGORITHMS  = ['Kyber512', 'Dilithium2', 'McEliece\n6960119', 'SPHINCS+\n256s']
ALG_SHORT   = ['Kyber512', 'Dilithium2', 'McEliece', 'SPHINCS+']
COLORS      = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff']  # blue, green, red, purple

print('Libraries loaded. Style configured.')

## 1 — Benchmark Data

Results collected from 100 iterations on Google Colab (CPU only).

In [ ]:
# ── Raw benchmark results ─────────────────────────────────────────────────────
# Times in milliseconds (average over 100 iterations)
# Sizes in bytes

data = {
    # Key / artifact sizes
    'public_key_bytes':  [800,    1_312,  1_047_319, 64     ],
    'secret_key_bytes':  [1_632,  2_528,  13_948,    128    ],
    'payload_bytes':     [768,    2_420,  194,        29_792 ],  # ciphertext or signature

    # Operation latency (ms)
    'keygen_ms':         [2.68,   8.92,   426.73,    248.78 ],
    'encap_sign_ms':     [3.76,   49.21,  1.51,      2985.68],
    'decap_verify_ms':   [5.46,   42.30,  119.88,    4.09   ],
}

# Algorithm metadata
meta = {
    'type':       ['KEM',       'Signature', 'KEM',        'Signature' ],
    'family':     ['Lattice',   'Lattice',   'Code-based', 'Hash-based'],
    'nist_level': [1,           2,           5,            5           ],
    'hard_problem': [
        'Module-LWE',
        'Module-LWE + Module-SIS',
        'Code Decoding (NP-hard)',
        'Hash Function Properties',
    ],
    'qrom_proof': ['Non-tight', 'Non-tight', 'Tight', 'ROM/QROM'],
    'tls_viable': ['Yes', 'Yes', 'No', 'No'],
}

print('Data loaded.')

## 2 — Performance Metrics Table

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.8))
ax.axis('off')

col_labels = [
    'Algorithm', 'Public Key\n(bytes)', 'Secret Key\n(bytes)',
    'Payload\n(bytes)', 'KeyGen\n(ms)', 'Encap/Sign\n(ms)', 'Decap/Verify\n(ms)'
]

table_data = [
    [
        ALG_SHORT[i],
        f"{data['public_key_bytes'][i]:,}",
        f"{data['secret_key_bytes'][i]:,}",
        f"{data['payload_bytes'][i]:,}",
        f"{data['keygen_ms'][i]:.2f}",
        f"{data['encap_sign_ms'][i]:.2f}",
        f"{data['decap_verify_ms'][i]:.2f}",
    ]
    for i in range(4)
]

tbl = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 2.0)

# Header styling
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor('#1f6feb')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Row styling — alternate + highlight extremes
extremes = {
    # (row, col): color  — highlight worst/best values
    (3, 1): '#3d1a1a',   # McEliece huge public key  (row index 3 = McEliece)
    (4, 5): '#3d1a1a',   # SPHINCS+ huge sign time
    (1, 1): '#1a3d1a',   # Kyber small public key
    (2, 5): '#1a3d1a',   # McEliece fast encap
}

for i in range(1, 5):
    bg = '#1c2128' if i % 2 == 0 else '#161b22'
    for j in range(len(col_labels)):
        cell = tbl[(i, j)]
        cell.set_facecolor(bg)
        cell.set_text_props(color='#c9d1d9')
        cell.set_edgecolor('#30363d')
    # Color the algorithm name with its dedicated color
    tbl[(i, 0)].set_facecolor(COLORS[i - 1] + '33')
    tbl[(i, 0)].set_text_props(color=COLORS[i - 1], fontweight='bold')

fig.patch.set_facecolor('#0d1117')
ax.set_title('Table 1 — Performance Metrics (avg. 100 iterations, Google Colab CPU)',
             pad=14, fontsize=12, color='#f0f6fc')
plt.tight_layout()
plt.savefig('table_performance.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → table_performance.png')

## 3 — Key & Payload Sizes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Key and Payload Sizes by Algorithm', fontsize=14,
             color='#f0f6fc', y=1.01)

x = np.arange(4)
bar_kw = dict(width=0.55, edgecolor='#0d1117', linewidth=0.8)

panels = [
    ('public_key_bytes', 'Public Key Size (bytes)', 'Public Key'),
    ('secret_key_bytes', 'Secret Key Size (bytes)', 'Secret Key'),
    ('payload_bytes',    'Payload Size (bytes)',    'Ciphertext / Signature'),
]

for ax, (key, ylabel, title) in zip(axes, panels):
    values = data[key]
    bars = ax.bar(x, values, color=COLORS, **bar_kw)

    # Value labels on top of bars
    for bar, val in zip(bars, values):
        label = f'{val:,}' if val < 100_000 else f'{val/1024:.0f} KB'
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            label,
            ha='center', va='bottom', fontsize=8.5, color='#c9d1d9'
        )

    ax.set_title(title, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(ALG_SHORT, fontsize=9)
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)

    # Log scale if range is too wide
    if max(values) / (min(v for v in values if v > 0) or 1) > 200:
        ax.set_yscale('log')
        ax.set_ylabel(ylabel + ' (log scale)', fontsize=10)

plt.tight_layout()
plt.savefig('chart_sizes.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → chart_sizes.png')

## 4 — Operation Latency

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Average Operation Latency (ms) — 100 iterations', fontsize=14,
             color='#f0f6fc', y=1.01)

panels = [
    ('keygen_ms',       'Time (ms)', 'Key Generation'),
    ('encap_sign_ms',   'Time (ms)', 'Encapsulation / Signing'),
    ('decap_verify_ms', 'Time (ms)', 'Decapsulation / Verification'),
]

for ax, (key, ylabel, title) in zip(axes, panels):
    values = data[key]
    bars = ax.bar(x, values, color=COLORS, **bar_kw)

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            f'{val:.2f} ms',
            ha='center', va='bottom', fontsize=8.5, color='#c9d1d9'
        )

    ax.set_title(title, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(ALG_SHORT, fontsize=9)
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)

    if max(values) / (min(v for v in values if v > 0) or 1) > 100:
        ax.set_yscale('log')
        ax.set_ylabel(ylabel + ' (log scale)', fontsize=10)

plt.tight_layout()
plt.savefig('chart_latency.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → chart_latency.png')

## 5 — Radar Chart: Overall Algorithm Profile

Each axis represents a normalized metric (higher = better for all axes).  
Inverses are applied to size and latency so that *smaller* values score higher.

In [ ]:
# Dimensions (all normalized 0→1, higher = better)
# For size/latency metrics, we invert: score = 1 - (val / max_val)
labels = [
    'Small\nPublic Key',
    'Small\nPayload',
    'Fast\nKeyGen',
    'Fast\nEncap/Sign',
    'Fast\nDecap/Verify',
    'Security\nLevel',
]
N = len(labels)

def normalize_inv(values):
    """Invert and normalize: smaller original value → higher score."""
    arr = np.array(values, dtype=float)
    return 1 - (arr - arr.min()) / (arr.max() - arr.min() + 1e-12)

def normalize(values):
    """Normalize: larger original value → higher score."""
    arr = np.array(values, dtype=float)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-12)

scores = np.column_stack([
    normalize_inv(data['public_key_bytes']),
    normalize_inv(data['payload_bytes']),
    normalize_inv(data['keygen_ms']),
    normalize_inv(data['encap_sign_ms']),
    normalize_inv(data['decap_verify_ms']),
    normalize(meta['nist_level']),
])  # shape: (4 algorithms, 6 dimensions)

# ── Radar plot ────────────────────────────────────────────────────────────────
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

for i, (alg, color) in enumerate(zip(ALG_SHORT, COLORS)):
    vals = scores[i].tolist() + scores[i][:1].tolist()
    ax.plot(angles, vals, color=color, linewidth=2, label=alg)
    ax.fill(angles, vals, color=color, alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=10, color='#c9d1d9')
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], size=7, color='#8b949e')
ax.set_ylim(0, 1)
ax.grid(color='#30363d', linewidth=0.8)
ax.spines['polar'].set_color('#30363d')

ax.set_title('Algorithm Profile — Normalized Scores\n(higher = better on each axis)',
             size=12, color='#f0f6fc', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.32, 1.15),
          facecolor='#161b22', edgecolor='#30363d',
          labelcolor='#c9d1d9', fontsize=10)

plt.tight_layout()
plt.savefig('chart_radar.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → chart_radar.png')

## 6 — TLS Handshake Overhead Simulation

Estimated bandwidth added to a TLS 1.3 handshake per connection.

In [ ]:
# TLS handshake overhead = public_key (sent by server) + payload (sent by client)
# For KEM (Kyber, McEliece): overhead = public_key + ciphertext
# For Signature (Dilithium, SPHINCS+): overhead = public_key + signature

tls_overhead_bytes = [
    data['public_key_bytes'][i] + data['payload_bytes'][i]
    for i in range(4)
]
tls_overhead_kb = [v / 1024 for v in tls_overhead_bytes]

# RSA-2048 and ECDH-P256 as classical baselines
classical = {
    'RSA-2048\n(classical)':    (256 + 256) / 1024,   # ~0.5 KB
    'ECDH-P256\n(classical)':   (65  + 65)  / 1024,   # ~0.13 KB
}

fig, ax = plt.subplots(figsize=(11, 5))

# PQC bars
bars_pqc = ax.bar(x, tls_overhead_kb, color=COLORS, width=0.5,
                  edgecolor='#0d1117', linewidth=0.8, label='PQC algorithms')

# Classical baselines as horizontal dashed lines
line_colors = ['#e3b341', '#79c0ff']
for (name, val), lc in zip(classical.items(), line_colors):
    ax.axhline(val, color=lc, linewidth=1.4, linestyle='--', alpha=0.85)
    ax.text(3.62, val + 0.008, name.replace('\n', ' '),
            color=lc, fontsize=8.5, va='bottom')

# Value labels
for bar, val_b, val_kb in zip(bars_pqc, tls_overhead_bytes, tls_overhead_kb):
    label = f'{val_b:,} B' if val_b < 10_000 else f'{val_kb:.0f} KB'
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            label, ha='center', va='bottom', fontsize=9, color='#c9d1d9')

ax.set_xticks(x)
ax.set_xticklabels(ALG_SHORT, fontsize=10)
ax.set_ylabel('Bandwidth Overhead (KB per connection)', fontsize=10)
ax.set_title('Estimated TLS Handshake Bandwidth Overhead\n'
             '(public key + ciphertext/signature — classical baselines shown as dashed lines)',
             fontsize=12, color='#f0f6fc')
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
ax.set_yscale('log')
ax.set_ylabel('Bandwidth Overhead (KB per connection, log scale)', fontsize=10)

plt.tight_layout()
plt.savefig('chart_tls_overhead.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

print('\nTLS overhead per connection:')
for alg, kb in zip(ALG_SHORT, tls_overhead_kb):
    print(f'  {alg:12s}: {kb:>8.2f} KB')
print(f'  {"RSA-2048":12s}: {classical["RSA-2048\n(classical)"]:.2f} KB (classical baseline)')
print(f'  {"ECDH-P256":12s}: {classical["ECDH-P256\n(classical)"]:.2f} KB (classical baseline)')
print('Saved → chart_tls_overhead.png')

## 7 — Security Metrics Table

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.6))
ax.axis('off')

sec_cols  = ['Algorithm', 'Type', 'Family', 'NIST Level', 'Hard Problem', 'QROM Proof', 'TLS Viable']
sec_data  = [
    [ALG_SHORT[i], meta['type'][i], meta['family'][i],
     str(meta['nist_level'][i]), meta['hard_problem'][i],
     meta['qrom_proof'][i], meta['tls_viable'][i]]
    for i in range(4)
]

tbl = ax.table(
    cellText=sec_data, colLabels=sec_cols,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
tbl.scale(1, 2.0)
tbl.auto_set_column_width(list(range(len(sec_cols))))

for j in range(len(sec_cols)):
    tbl[(0, j)].set_facecolor('#1f6feb')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

viable_colors = {'Yes': '#1a3d1a', 'No': '#3d1a1a'}

for i in range(1, 5):
    bg = '#1c2128' if i % 2 == 0 else '#161b22'
    for j in range(len(sec_cols)):
        cell = tbl[(i, j)]
        cell.set_facecolor(bg)
        cell.set_text_props(color='#c9d1d9')
        cell.set_edgecolor('#30363d')
    # Algorithm name color
    tbl[(i, 0)].set_facecolor(COLORS[i-1] + '33')
    tbl[(i, 0)].set_text_props(color=COLORS[i-1], fontweight='bold')
    # TLS viability color
    v = meta['tls_viable'][i-1]
    tbl[(i, 6)].set_facecolor(viable_colors[v])
    tbl[(i, 6)].set_text_props(
        color='#3fb950' if v == 'Yes' else '#f78166', fontweight='bold'
    )

fig.patch.set_facecolor('#0d1117')
ax.set_title('Table 2 — Security Metrics Summary', pad=14,
             fontsize=12, color='#f0f6fc')
plt.tight_layout()
plt.savefig('table_security.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → table_security.png')

## 8 — Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

ax_pk   = fig.add_subplot(gs[0, 0])
ax_sig  = fig.add_subplot(gs[0, 1])
ax_kg   = fig.add_subplot(gs[0, 2])
ax_es   = fig.add_subplot(gs[1, 0])
ax_dv   = fig.add_subplot(gs[1, 1])
ax_tls  = fig.add_subplot(gs[1, 2])

panels_dash = [
    (ax_pk,  data['public_key_bytes'],  'Public Key Size (bytes)',         True),
    (ax_sig, data['payload_bytes'],     'Payload Size (bytes)',             True),
    (ax_kg,  data['keygen_ms'],         'KeyGen Latency (ms)',              True),
    (ax_es,  data['encap_sign_ms'],     'Encap / Sign Latency (ms)',        True),
    (ax_dv,  data['decap_verify_ms'],   'Decap / Verify Latency (ms)',      True),
    (ax_tls, tls_overhead_kb,           'TLS Overhead (KB per connection)', True),
]

for ax, values, title, use_log in panels_dash:
    bars = ax.bar(x, values, color=COLORS, width=0.6,
                  edgecolor='#0d1117', linewidth=0.7)
    ax.set_title(title, fontsize=9.5, pad=6)
    ax.set_xticks(x)
    ax.set_xticklabels(ALG_SHORT, fontsize=8)
    ax.yaxis.grid(True, alpha=0.35)
    ax.set_axisbelow(True)
    if use_log and max(values) / (min(v for v in values if v > 0) or 1) > 50:
        ax.set_yscale('log')

legend_patches = [
    mpatches.Patch(facecolor=c, label=a)
    for c, a in zip(COLORS, ALG_SHORT)
]
fig.legend(handles=legend_patches, loc='upper center',
           ncol=4, facecolor='#161b22', edgecolor='#30363d',
           labelcolor='#c9d1d9', fontsize=10,
           bbox_to_anchor=(0.5, 1.02))

fig.suptitle('Post-Quantum Cryptography — Performance Dashboard',
             fontsize=15, color='#f0f6fc', y=1.06)

plt.savefig('dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → dashboard.png')

## 9 — Conclusions

The benchmarks confirm the trade-off landscape described in the research report:

**Lattice-based algorithms (Kyber, Dilithium)** deliver the best overall balance — compact keys, fast operations, and strong provable security. They are the natural default for latency-sensitive protocols such as TLS/HTTPS.

**Classic McEliece** offers the deepest security pedigree (~50 years of cryptanalysis resistance) and a tight QROM proof, but its ~1 MB public key renders it impractical for most network applications.

**SPHINCS+** makes the most conservative security assumption (hash-function properties only) and is ideal for infrequent, high-assurance signatures — firmware signing, root CA certificates, cold-storage attestations — where a 3-second signing time is an acceptable cost for maximum theoretical trust.

The NIST's deliberate choice to standardize all four families is itself a risk-mitigation strategy: if a theoretical advance ever breaks one mathematical family, the others remain unaffected.

---
*Generated charts: `table_performance.png`, `chart_sizes.png`, `chart_latency.png`, `chart_radar.png`, `chart_tls_overhead.png`, `table_security.png`, `dashboard.png`*